# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [5]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [6]:
highlight_country = "China"
highlight_color = "#0072B2"

fig = go.Figure()

# Show every other country in faint grey
for country in df["Country"].unique():
    if country == highlight_country:
        continue

    country_data = df[df["Country"] == country].sort_values("Year")
    fig.add_trace(go.Scatter(
        x=country_data["Year"],
        y=country_data["CO2_per_capita"],
        mode="lines",
        line=dict(color="#EEEEEE", width=1),
        showlegend=False,
        hovertemplate=f"{country}<br>Year: %{{x}}<br>CO2 per capita: %{{y}}<extra></extra>",
    ))

# Draw China on top so it stands out
highlighted = (
    df[df["Country"] == highlight_country]
    .sort_values("Year")
)

fig.add_trace(go.Scatter(
    x=highlighted["Year"],
    y=highlighted["CO2_per_capita"],
    mode="lines",
    line=dict(color=highlight_color, width=3),
    showlegend=False,
    hovertemplate="China<br>Year: %{x}<br>CO2 per capita: %{y}<extra></extra>",
))

# Put the label beside China's final data point
fig.add_annotation(
    x=highlighted["Year"].iloc[-1],
    y=highlighted["CO2_per_capita"].iloc[-1],
    text=highlight_country,
    showarrow=False,
    xshift=8,
    xanchor="left",
    font=dict(color=highlight_color, size=13),
)

fig.update_layout(
    title="China's CO2 per capita climbed from 1.32 to 4.37, 2000-2022",
    xaxis_title="Year",
    yaxis_title="CO2 per capita",
    template="plotly_white",
    showlegend=False,
    margin=dict(r=100),
)

# Leave room after the final year for the direct label
fig.update_xaxes(
    range=[df["Year"].min(), df["Year"].max() + 3],
    showline=False,
    zeroline=False,
    gridcolor="#F2F2F2",
)
fig.update_yaxes(
    showline=False,
    zeroline=False,
    gridcolor="#F2F2F2",
)

%pip install "nbformat>=4.2.0"
fig.show()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [9]:
# Task — Multi-series line with highlight
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('../data/co2_emissions.csv')

HIGHLIGHT = 'China'
asia = df[df['Region'] == 'Asia']

fig = go.Figure()

# Other Asian countries: thin grey lines
for country, d in asia.groupby('Country'):
    if country != HIGHLIGHT:
        fig.add_trace(go.Scatter(x=d['Year'], y=d['CO2_Mt'], mode='lines',
                                 line=dict(color='#EEEEEE', width=1),
                                 showlegend=False))

# Highlighted country: thick red line
h = asia[asia['Country'] == HIGHLIGHT].sort_values('Year')
fig.add_trace(go.Scatter(x=h['Year'], y=h['CO2_Mt'], mode='lines',
                         line=dict(color='#D62728', width=4),
                         showlegend=False))

# Direct label at the end of the line
fig.add_annotation(x=h['Year'].iloc[-1], y=h['CO2_Mt'].iloc[-1],
                   text=f'<b>{HIGHLIGHT}</b>', showarrow=False,
                   xanchor='left', xshift=8,
                   font=dict(color='#D62728', size=14))

# Insight title based on the data
change = (h['CO2_Mt'].iloc[-1] / h['CO2_Mt'].iloc[0] - 1) * 100
fig.update_layout(
    title=f"<b>{HIGHLIGHT}'s CO2 emissions grew {change:.0f}% since 2000, "
          f"far ahead of other Asian countries</b>",
    template='simple_white',
    xaxis=dict(title='', showgrid=False, showline=False, tickangle=0),
    yaxis=dict(title='CO2 emissions (Mt)', showgrid=False, showline=False),
    margin=dict(r=110), width=900, height=500)

fig.show()